In [ ]:
import pandas as pd
import ast
import numpy as np

df = pd.read_csv("Kickstarter_processed.csv")

print("Kickstarter shape:", df.shape)
print(df.columns.tolist())

def parse_tokens(x):
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x
    
    if isinstance(x, str):
        x = x.strip()
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except:
            return x.split()
    
    return []

df["tokens"] = df["description_processed"].apply(parse_tokens)

print("\nExample tokens:")
print(df["tokens"].iloc[0][:20])

conc_df = pd.read_csv("concreteness_lexicon.csv")

print("\nLexicon columns:")
print(conc_df.columns.tolist())

print("\nLexicon preview:")
print(conc_df.head())

concreteness_dict = dict(
    zip(
        conc_df["Word"].astype(str).str.lower(),
        conc_df["Conc.M"]
    )
)

print("\nExample lookup:")
sample_words = ["apple", "freedom", "music", "project"]
for w in sample_words:
    print(w, "->", concreteness_dict.get(w, "not found"))


def compute_concreteness(tokens, lexicon):
    if not tokens:
        return np.nan
    
    scores = [
        lexicon[token.lower()]
        for token in tokens
        if token.lower() in lexicon
    ]
    
    if len(scores) == 0:
        return np.nan
    
    return np.mean(scores)

df["concreteness_score"] = df["tokens"].apply(
    lambda toks: compute_concreteness(toks, concreteness_dict)
)

def compute_coverage(tokens, lexicon):
    if not tokens:
        return 0.0
    
    matched = sum(1 for token in tokens if token.lower() in lexicon)
    return matched / len(tokens)

df["concreteness_coverage"] = df["tokens"].apply(
    lambda toks: compute_coverage(toks, concreteness_dict)
)


print("\nConcreteness summary:")
print(df["concreteness_score"].describe())

print("\nCoverage summary:")
print(df["concreteness_coverage"].describe())

print("\nSample output:")
print(df[[
    "title",
    "concreteness_score",
    "concreteness_coverage"
]].head(10))

df.to_csv("Kickstarter_with_concreteness.csv", index=False)
print("\nSaved file: Kickstarter_with_concreteness.csv")

Kickstarter shape: (7354, 13)
['Unnamed: 0', 'url', 'title', 'description', 'pledged', 'usd_pledged', 'converted_pledged_amount', 'goal', 'currency', 'category', 'reached', 'status', 'description_processed']

Example tokens:
['entertainment', 'today', 'push', 'unholy', 'agenda', 'consume', 'guard', 'sin', 'convince', 'adopt', 'distract', 'actually', 'matter', 'streaming', 'network', 'streaming', 'service', 'inspire', 'truth', 'subscription']

Lexicon columns:
['Word', 'Bigram', 'Conc.M', 'Conc.SD', 'Unknown', 'Total', 'Percent_known', 'SUBTLEX', 'Dom_Pos']

Lexicon preview:
            Word  Bigram  Conc.M  Conc.SD  Unknown  Total  Percent_known  \
0    roadsweeper       0    4.85     0.37        1     27           0.96   
1    traindriver       0    4.54     0.71        3     29           0.90   
2           tush       0    4.45     1.01        3     25           0.88   
3      hairdress       0    3.93     1.28        0     29           1.00   
4  pharmaceutics       0    3.77     1.